# Numerical experiments for noise-aware WRPGA stopping

This notebook reproduces the numerical experiments accompanying the manuscript on noise-aware stopping for the Weak Rescaled Pure Greedy Algorithm (WRPGA).

The experiments are designed to test four statements qualitatively and quantitatively:

1. residual discrepancy stopping produces reconstruction error at the deterministic noise scale;
2. the correlation stopping rule is support-preserving in the incoherent sparse regime;
3. the beta-min condition predicts a safe region for full support coverage;
4. WRPGA is compared with OMP/OGA using the same dictionary search but only scalar rescaling instead of a growing least-squares projection.

**Important interpretation.** The theorem gives a worst-case upper bound of order $O(\delta^{-2})$ on the stopping index. It does not imply that Monte Carlo averages must scale exactly as $\delta^{-2}$. We therefore report empirical log-log slopes but interpret them only descriptively.

In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 20260813
rng = np.random.default_rng(SEED)

OUT = Path('/mnt/data/wrpga_numerics')
OUT.mkdir(parents=True, exist_ok=True)

print(f'Output directory: {OUT}')
print(f'Random seed: {SEED}')

## Core constructions

For the incoherence experiments we use an equicorrelated dictionary
\[
 g_j=\sqrt{1-\mu}\,e_j+\sqrt{\mu}\,e_0,
\]
for which every atom has unit norm and every pair of distinct atoms has inner product exactly $\mu$. Thus the mutual coherence is known exactly rather than estimated from a random matrix.

In [ ]:
def equicorrelated_dictionary(p, mu):
    """Return D in R^{(p+1) x p} with unit columns and mutual coherence exactly mu."""
    if not (0 <= mu < 1):
        raise ValueError('mu must lie in [0,1).')
    D = np.zeros((p + 1, p), dtype=float)
    D[0, :] = np.sqrt(mu)
    D[1:, :] = np.sqrt(1 - mu) * np.eye(p)
    return D




def adversarial_coherent_dictionary(p, s, mu):
    """Dictionary with orthonormal true support atoms and one coherent distractor.

    Columns 0,...,s-1 are the true support atoms e_1,...,e_s. Column s has
    inner product mu with each support atom. Remaining columns are orthogonal
    distractors. The mutual coherence is exactly mu provided s*mu**2 < 1.
    """
    if s * mu**2 >= 1:
        raise ValueError('Need s*mu^2 < 1 for the adversarial atom construction.')
    n = p + 1
    D = np.zeros((n, p), dtype=float)
    D[np.arange(s), np.arange(s)] = 1.0
    D[:s, s] = mu
    D[s, s] = np.sqrt(1 - s * mu**2)
    for j in range(s + 1, p):
        D[j, j] = 1.0
    return D

def mutual_coherence(D):
    G = D.T @ D
    np.fill_diagonal(G, 0.0)
    return float(np.max(np.abs(G)))


def add_deterministic_norm_noise(clean, delta, rng):
    z = rng.normal(size=clean.size)
    nz = np.linalg.norm(z)
    if nz == 0:
        z[0] = 1.0
        nz = 1.0
    eta = delta * z / nz
    return clean + eta, eta


def coherence_constants(s, mu, t_star=1.0):
    L = 1 - (s - 1) * mu
    if L <= 0:
        return dict(L=L, A=np.nan, C=np.nan, Delta=np.nan, G=np.nan, theta_min=np.nan,
                    beta_ratio=np.nan, mu_threshold=t_star / (s + t_star * (s - 1)))
    A = np.sqrt(L / s)
    C = mu * np.sqrt(s) / np.sqrt(L)
    Delta = t_star * A - C
    G = (1 + t_star) / Delta if Delta > 0 else np.inf
    theta_min = 1 + G
    beta_ratio = None
    return dict(L=L, A=A, C=C, Delta=Delta, G=G, theta_min=theta_min,
                mu_threshold=t_star / (s + t_star * (s - 1)))

## Algorithms

The WRPGA implementation follows the manuscript exactly:
\[
\lambda_m=\langle r_{m-1},\varphi_m\rangle,\qquad
\widehat u_m=u_{m-1}+\lambda_m\varphi_m,
\]
followed by the scalar rescaling
\[
s_m=\frac{\langle f,\widehat u_m\rangle}{\|\widehat u_m\|^2},\qquad
u_m=s_m\widehat u_m.
\]

OMP uses the same maximum-correlation atom selection but recomputes a least-squares projection on the span of all distinct selected atoms.

In [ ]:
def wrpga(D, f, delta=None, kappa=None, theta=None, max_iter=10000):
    n, p = D.shape
    u = np.zeros(n)
    r = f.copy()
    selected = []
    hist = []

    for m in range(max_iter + 1):
        corr = D.T @ r
        nu = float(np.max(np.abs(corr)))
        residual_norm = float(np.linalg.norm(r))
        hist.append((m, residual_norm, nu))

        stop_res = delta is not None and kappa is not None and residual_norm <= kappa * delta
        stop_corr = delta is not None and theta is not None and nu <= theta * delta
        if stop_res or stop_corr:
            return {
                'u': u, 'r': r, 'selected': selected,
                'history': np.asarray(hist),
                'stop_reason': 'residual' if stop_res else 'correlation',
                'iterations': m,
            }

        j = int(np.argmax(np.abs(corr)))
        lam = float(corr[j])
        selected.append(j)

        uhat = u + lam * D[:, j]
        den = float(uhat @ uhat)
        if den == 0.0:
            u = np.zeros_like(u)
        else:
            sm = float(f @ uhat) / den
            u = sm * uhat
        r = f - u

    return {'u': u, 'r': r, 'selected': selected, 'history': np.asarray(hist),
            'stop_reason': 'max_iter', 'iterations': max_iter}


def omp(D, f, delta=None, kappa=None, theta=None, max_iter=None):
    n, p = D.shape
    if max_iter is None:
        max_iter = min(n, p)
    u = np.zeros(n)
    r = f.copy()
    support = []
    hist = []

    for m in range(max_iter + 1):
        corr = D.T @ r
        nu = float(np.max(np.abs(corr)))
        residual_norm = float(np.linalg.norm(r))
        hist.append((m, residual_norm, nu))

        stop_res = delta is not None and kappa is not None and residual_norm <= kappa * delta
        stop_corr = delta is not None and theta is not None and nu <= theta * delta
        if stop_res or stop_corr:
            return {'u': u, 'r': r, 'selected': support.copy(), 'history': np.asarray(hist),
                    'stop_reason': 'residual' if stop_res else 'correlation', 'iterations': m}

        order = np.argsort(-np.abs(corr))
        j = next((int(jj) for jj in order if int(jj) not in support), None)
        if j is None:
            return {'u': u, 'r': r, 'selected': support.copy(), 'history': np.asarray(hist),
                    'stop_reason': 'stagnation', 'iterations': m}
        support.append(j)
        X = D[:, support]
        coef, *_ = np.linalg.lstsq(X, f, rcond=None)
        u = X @ coef
        r = f - u

    return {'u': u, 'r': r, 'selected': support.copy(), 'history': np.asarray(hist),
            'stop_reason': 'max_iter', 'iterations': max_iter}


def pga(D, f, delta=None, kappa=None, theta=None, max_iter=10000):
    u = np.zeros_like(f)
    r = f.copy()
    selected = []
    hist = []
    for m in range(max_iter + 1):
        corr = D.T @ r
        nu = float(np.max(np.abs(corr)))
        residual_norm = float(np.linalg.norm(r))
        hist.append((m, residual_norm, nu))
        stop_res = delta is not None and kappa is not None and residual_norm <= kappa * delta
        stop_corr = delta is not None and theta is not None and nu <= theta * delta
        if stop_res or stop_corr:
            return {'u': u, 'r': r, 'selected': selected, 'history': np.asarray(hist),
                    'stop_reason': 'residual' if stop_res else 'correlation', 'iterations': m}
        j = int(np.argmax(np.abs(corr)))
        lam = float(corr[j])
        selected.append(j)
        u = u + lam * D[:, j]
        r = f - u
    return {'u': u, 'r': r, 'selected': selected, 'history': np.asarray(hist),
            'stop_reason': 'max_iter', 'iterations': max_iter}

## Sanity checks

These checks verify the exact coherence of the synthetic dictionary and the WRPGA orthogonality/descent identities numerically.

In [ ]:
D_test = equicorrelated_dictionary(32, 0.05)
assert np.allclose(np.linalg.norm(D_test, axis=0), 1.0)
assert np.isclose(mutual_coherence(D_test), 0.05)

clean = D_test[:, [1, 5, 9, 14]] @ np.array([1.5, -1.2, 1.8, -1.1])
f_test, _ = add_deterministic_norm_noise(clean, 0.05, rng)
res = wrpga(D_test, f_test, delta=0.05, kappa=1.2, max_iter=500)
assert abs(res['r'] @ res['u']) < 1e-9
print('Sanity checks passed.')

# Experiment 1: residual discrepancy and the noise scale

We use an orthonormal dictionary and a compressible coefficient vector $c_j=1/j$. This is deliberately harder than an exactly sparse vector and makes the stopping index grow visibly as the noise level decreases.

We report:
- the stopping iteration $m_\delta$;
- the clean reconstruction error $\|f^\delta-u_{m_\delta}\|$;
- the normalized error $\|f^\delta-u_{m_\delta}\|/\delta$;
- an empirical log-log slope for the stopping index.

The theoretical $O(\delta^{-2})$ statement is an upper bound, so an empirical slope with magnitude smaller than $2$ is fully consistent with the theorem.

In [ ]:
def wrpga_identity(f, delta, kappa=1.2, max_iter=None):
    # Same WRPGA recursion for D=I, exploiting D^T r = r without forming I.
    if max_iter is None:
        max_iter = f.size
    u = np.zeros_like(f)
    r = f.copy()
    for m in range(max_iter + 1):
        if np.linalg.norm(r) <= kappa * delta:
            return {'u': u, 'r': r, 'iterations': m}
        j = int(np.argmax(np.abs(r)))
        lam = float(r[j])
        uhat = u.copy()
        uhat[j] += lam
        den = float(uhat @ uhat)
        u = (float(f @ uhat) / den) * uhat if den > 0 else np.zeros_like(u)
        r = f - u
    return {'u': u, 'r': r, 'iterations': max_iter}


def experiment_residual_rate(n=2048, deltas=None, trials=20, kappa=1.2):
    if deltas is None:
        deltas = np.array([0.20, 0.15, 0.10, 0.075, 0.050, 0.035, 0.025, 0.018, 0.0125])
    clean = 1.0 / np.arange(1, n + 1, dtype=float)
    rows = []
    local_rng = np.random.default_rng(SEED + 1)

    for delta in deltas:
        for trial in range(trials):
            f, _ = add_deterministic_norm_noise(clean, delta, local_rng)
            out = wrpga_identity(f, delta=delta, kappa=kappa, max_iter=n)
            rows.append({
                'delta': delta,
                'trial': trial,
                'iterations': out['iterations'],
                'clean_error': np.linalg.norm(clean - out['u']),
                'normalized_error': np.linalg.norm(clean - out['u']) / delta,
                'residual_norm': np.linalg.norm(out['r']),
            })
    return pd.DataFrame(rows), clean

rate_df, rate_clean = experiment_residual_rate()
rate_summary = rate_df.groupby('delta').agg(
    mean_iterations=('iterations', 'mean'),
    sd_iterations=('iterations', 'std'),
    mean_clean_error=('clean_error', 'mean'),
    sd_clean_error=('clean_error', 'std'),
    mean_normalized_error=('normalized_error', 'mean'),
).reset_index()

slope_m, intercept_m = np.polyfit(np.log(rate_summary['delta']), np.log(rate_summary['mean_iterations']), 1)
slope_e, intercept_e = np.polyfit(np.log(rate_summary['delta']), np.log(rate_summary['mean_clean_error']), 1)
print(rate_summary)
print(f'Empirical stopping-index slope: {slope_m:.3f}')
print(f'Empirical clean-error slope: {slope_e:.3f}')
rate_df.to_csv(OUT / 'experiment1_residual_trials.csv', index=False)
rate_summary.to_csv(OUT / 'experiment1_residual_summary.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.errorbar(rate_summary['delta'], rate_summary['mean_iterations'],
            yerr=rate_summary['sd_iterations'], marker='o', capsize=3)
ref_delta = rate_summary['delta'].to_numpy()
ref = rate_summary['mean_iterations'].iloc[0] * (ref_delta / ref_delta[0])**(-2)
ax.plot(ref_delta, ref, '--', label=r'reference slope $-2$')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(r'noise level $\delta$')
ax.set_ylabel(r'mean stopping iteration $m_\delta$')
ax.set_title('Residual discrepancy stopping')
ax.legend()
fig.tight_layout()
fig.savefig(OUT / 'fig_residual_iterations.pdf', bbox_inches='tight')
fig.savefig(OUT / 'fig_residual_iterations.png', dpi=220, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(rate_summary['delta'], rate_summary['mean_normalized_error'], marker='o')
ax.axhline(2.2, linestyle='--', label=r'theoretical bound $1+\kappa=2.2$')
ax.set_xscale('log')
ax.set_xlabel(r'noise level $\delta$')
ax.set_ylabel(r'mean $\|f^\delta-u_{m_\delta}\|_2/\delta$')
ax.set_title('Normalized clean-signal error at residual stopping')
ax.legend()
fig.tight_layout()
fig.savefig(OUT / 'fig_residual_error.pdf', bbox_inches='tight')
fig.savefig(OUT / 'fig_residual_error.png', dpi=220, bbox_inches='tight')
plt.show()

# Experiment 2: support preservation versus coherence

For full greediness $t_*=1$, the theorem assumes
\[
\mu<\frac{1}{2s-1}.
\]
We use a deliberately adversarial dictionary containing one distractor atom with correlation $\mu$ with every true support atom, and vary $\mu$ across the structural threshold. A fixed correlation parameter $\vartheta$ is used for comparability; the plot marks both $1/(2s-1)$ and the largest $\mu$ for which this fixed $\vartheta$ satisfies all theorem hypotheses. The conditions are sufficient, so no sharp empirical phase transition is expected at either line.

In [ ]:
def experiment_support_vs_coherence(p=96, s=4, delta=0.005, mus=None, trials=150, theta=160.0):
    if mus is None:
        mus = np.array([0.00, 0.05, 0.10, 0.12, 0.135, 0.14, 0.15, 0.20, 0.28, 0.35, 0.45])
    local_rng = np.random.default_rng(SEED + 2)
    rows=[]
    threshold = 1 / (2*s - 1)

    # For the fixed theta used in this experiment, compute the largest tested mu
    # for which all theorem conditions (including theta > 1+G_*) are certified.
    grid = np.linspace(0, threshold * (1 - 1e-6), 20000)
    certified = []
    for x in grid:
        c = coherence_constants(s, x, 1.0)
        certified.append(np.isfinite(c['theta_min']) and theta > c['theta_min'])
    certified_mu = grid[np.where(certified)[0][-1]] if any(certified) else 0.0

    for mu in mus:
        D = adversarial_coherent_dictionary(p, s, mu)
        assert np.isclose(mutual_coherence(D), mu, atol=1e-12)
        const = coherence_constants(s, mu, 1.0)
        theorem_guaranteed = (mu < threshold and np.isfinite(const['theta_min'])
                              and theta > const['theta_min'])
        S = np.arange(s)
        for trial in range(trials):
            amps = local_rng.uniform(1.0, 2.0, size=s) * local_rng.choice([-1.0, 1.0], size=s)
            clean = D[:, S] @ amps
            f, _ = add_deterministic_norm_noise(clean, delta, local_rng)
            out = wrpga(D, f, delta=delta, theta=theta, max_iter=2000)
            selected = np.asarray(out['selected'], dtype=int)
            false = bool(np.any(~np.isin(selected, S))) if selected.size else False
            covered = set(S).issubset(set(selected.tolist()))
            rows.append({
                'mu': mu, 'trial': trial, 'theta': theta,
                'iterations': out['iterations'], 'false_selection': false,
                'full_support_covered': covered,
                'clean_error': np.linalg.norm(clean - out['u']),
                'normalized_error': np.linalg.norm(clean - out['u']) / delta,
                'inside_structural_coherence_region': mu < threshold,
                'theorem_guaranteed_for_fixed_theta': theorem_guaranteed,
            })
    return pd.DataFrame(rows), threshold, certified_mu

support_df, mu_threshold, certified_mu = experiment_support_vs_coherence()
support_summary = support_df.groupby('mu').agg(
    false_selection_rate=('false_selection', 'mean'),
    support_coverage_rate=('full_support_covered', 'mean'),
    mean_iterations=('iterations', 'mean'),
    mean_normalized_error=('normalized_error', 'mean'),
).reset_index()
print(f'Structural coherence threshold 1/(2s-1): {mu_threshold:.6f}')
print(f'Certified boundary for the fixed theta used here: {certified_mu:.6f}')
print(support_summary)
support_df.to_csv(OUT / 'experiment2_support_trials.csv', index=False)
support_summary.to_csv(OUT / 'experiment2_support_summary.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(support_summary['mu'], support_summary['false_selection_rate'], marker='o')
ax.axvline(mu_threshold, linestyle='--', label=r'structural threshold $1/(2s-1)$')
ax.axvline(certified_mu, linestyle=':', label=r'fixed-$\vartheta$ certified boundary')
ax.set_xlabel(r'mutual coherence $\mu$')
ax.set_ylabel('fraction with a false selection')
ax.set_ylim(-0.03, 1.03)
ax.set_title('False selections before correlation stopping')
ax.legend()
fig.tight_layout()
fig.savefig(OUT / 'fig_false_selection_vs_mu.pdf', bbox_inches='tight')
fig.savefig(OUT / 'fig_false_selection_vs_mu.png', dpi=220, bbox_inches='tight')
plt.show()

# Experiment 3: beta-min and full support coverage

For fixed $s,\mu$ and the theorem-safe correlation threshold $\vartheta\delta$, the beta-min corollary requires
\[
\frac{a_{\min}}{\delta}>
\frac{(\vartheta+1)\sqrt{s}}{1-(s-1)\mu}.
\]
We vary the signal-to-noise coefficient ratio $a_{\min}/\delta$ and measure the fraction of runs in which all true support atoms have been selected by termination.

In [ ]:
def experiment_beta_min(p=96, s=4, mu=0.05, delta=0.05,
                        ratios=None, trials=150, theta_margin=0.5):
    if ratios is None:
        ratios = np.array([2, 4, 6, 8, 10, 12, 15, 18, 20, 24, 28, 32], dtype=float)
    local_rng = np.random.default_rng(SEED + 3)
    D = equicorrelated_dictionary(p, mu)
    const = coherence_constants(s, mu, 1.0)
    theta = const['theta_min'] + theta_margin
    L = const['L']
    beta_threshold = (theta + 1) * np.sqrt(s) / L
    rows=[]

    for ratio in ratios:
        amin = ratio * delta
        for trial in range(trials):
            S = np.sort(local_rng.choice(p, size=s, replace=False))
            # All coefficients have magnitude at least amin, with modest random spread.
            mags = amin * local_rng.uniform(1.0, 1.25, size=s)
            amps = mags * local_rng.choice([-1.0, 1.0], size=s)
            clean = D[:, S] @ amps
            f, _ = add_deterministic_norm_noise(clean, delta, local_rng)
            out = wrpga(D, f, delta=delta, theta=theta, max_iter=3000)
            selected_set = set(out['selected'])
            full = set(S).issubset(selected_set)
            false = any(j not in set(S) for j in selected_set)
            rows.append({'ratio': ratio, 'trial': trial, 'full_support': full,
                         'false_selection': false, 'iterations': out['iterations'],
                         'normalized_error': np.linalg.norm(clean - out['u']) / delta})
    return pd.DataFrame(rows), beta_threshold, theta

beta_df, beta_threshold, beta_theta = experiment_beta_min()
beta_summary = beta_df.groupby('ratio').agg(
    support_recovery_rate=('full_support', 'mean'),
    false_selection_rate=('false_selection', 'mean'),
    mean_iterations=('iterations', 'mean'),
    mean_normalized_error=('normalized_error', 'mean'),
).reset_index()
print(f'Correlation parameter theta = {beta_theta:.4f}')
print(f'Theoretical beta-min ratio a_min/delta > {beta_threshold:.4f}')
print(beta_summary)
beta_df.to_csv(OUT / 'experiment3_beta_trials.csv', index=False)
beta_summary.to_csv(OUT / 'experiment3_beta_summary.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.plot(beta_summary['ratio'], beta_summary['support_recovery_rate'], marker='o')
ax.axvline(beta_threshold, linestyle='--', label='theoretical beta-min threshold')
ax.set_xlabel(r'$a_{\min}/\delta$')
ax.set_ylabel('full support coverage rate')
ax.set_ylim(-0.03, 1.03)
ax.set_title('Beta-min experiment')
ax.legend()
fig.tight_layout()
fig.savefig(OUT / 'fig_beta_min.pdf', bbox_inches='tight')
fig.savefig(OUT / 'fig_beta_min.png', dpi=220, bbox_inches='tight')
plt.show()

# Experiment 4: WRPGA versus OMP/OGA

We compare WRPGA and OMP under the same sparse signal model and the same correlation threshold. The main algorithmic distinction is that WRPGA performs only a scalar rescaling, while OMP solves a growing least-squares problem after each new atom.

The timing numbers are implementation- and hardware-dependent. They should be reported as descriptive computational evidence, not as an asymptotic complexity theorem.

In [ ]:
def experiment_algorithm_comparison(p=1024, s=16, mu=0.01, delta=0.01,
                                    trials=20, theta_margin=0.5, kappa=1.2):
    local_rng = np.random.default_rng(SEED + 4)
    D = equicorrelated_dictionary(p, mu)
    const = coherence_constants(s, mu, 1.0)
    if const['Delta'] <= 0:
        raise ValueError('Choose mu inside the theorem region for the comparison.')
    theta = const['theta_min'] + theta_margin
    rows=[]
    methods = [
        ('WRPGA-correlation', wrpga, {'theta': theta}),
        ('OMP-correlation', omp, {'theta': theta}),
        ('WRPGA-residual', wrpga, {'kappa': kappa}),
        ('OMP-residual', omp, {'kappa': kappa}),
    ]

    for trial in range(trials):
        S = np.sort(local_rng.choice(p, size=s, replace=False))
        amps = local_rng.uniform(1.0, 2.0, size=s) * local_rng.choice([-1.0, 1.0], size=s)
        clean = D[:, S] @ amps
        f, _ = add_deterministic_norm_noise(clean, delta, local_rng)
        for name, alg, kwargs in methods:
            t0 = time.perf_counter()
            out = alg(D, f, delta=delta, max_iter=3000, **kwargs)
            elapsed = time.perf_counter() - t0
            selected_set = set(out['selected'])
            false = any(j not in set(S) for j in selected_set)
            covered = set(S).issubset(selected_set)
            rows.append({
                'method': name, 'trial': trial,
                'iterations': out['iterations'],
                'runtime_seconds': elapsed,
                'clean_error': np.linalg.norm(clean - out['u']),
                'normalized_error': np.linalg.norm(clean - out['u']) / delta,
                'false_selection': false,
                'full_support_covered': covered,
            })
    return pd.DataFrame(rows), theta

comp_df, comp_theta = experiment_algorithm_comparison()
comp_summary = comp_df.groupby('method').agg(
    mean_iterations=('iterations', 'mean'),
    sd_iterations=('iterations', 'std'),
    mean_runtime_seconds=('runtime_seconds', 'mean'),
    median_runtime_seconds=('runtime_seconds', 'median'),
    mean_normalized_error=('normalized_error', 'mean'),
    false_selection_rate=('false_selection', 'mean'),
    support_coverage_rate=('full_support_covered', 'mean'),
).reset_index()
print(f'Comparison theta = {comp_theta:.4f}')
print(comp_summary.to_string(index=False))
comp_df.to_csv(OUT / 'experiment4_comparison_trials.csv', index=False)
comp_summary.to_csv(OUT / 'experiment4_comparison_summary.csv', index=False)

In [ ]:
# LaTeX-ready table
latex_table = comp_summary.copy()
for c in ['mean_iterations', 'mean_runtime_seconds', 'median_runtime_seconds',
          'mean_normalized_error', 'false_selection_rate', 'support_coverage_rate']:
    latex_table[c] = latex_table[c].map(lambda x: f'{x:.4g}')
tex = latex_table.to_latex(index=False, escape=False,
                           caption='Monte Carlo comparison of WRPGA and OMP stopping rules.',
                           label='tab:numerical-comparison')
(OUT / 'table_algorithm_comparison.tex').write_text(tex)
print(tex)

# Reproducibility summary

The following cell writes a compact text summary of all numerical settings and fitted slopes. The manuscript can cite the archived notebook/repository containing this file; the paper itself should state the random seed, dimensions, trial counts, noise construction, stopping parameters and dictionary construction.

In [ ]:
summary_text = (
    f'Numerical experiment reproducibility summary\n'
    f'Random seed: {SEED}\n'
    f'Experiment 1 empirical stopping-index slope: {slope_m:.6f}\n'
    f'Experiment 1 empirical clean-error slope: {slope_e:.6f}\n'
    f'Experiment 2 structural coherence threshold: {mu_threshold:.8f}\nExperiment 2 fixed-theta certified boundary: {certified_mu:.8f}\n'
    f'Experiment 3 theta: {beta_theta:.8f}\n'
    f'Experiment 3 beta-min threshold a_min/delta: {beta_threshold:.8f}\n'
    f'Experiment 4 theta: {comp_theta:.8f}\n'
)
(OUT / 'reproducibility_summary.txt').write_text(summary_text)
print(summary_text)